# 00 · Setup and data download

Run this notebook once before the workshop (ideally the day before, on the venue network).
It installs packages, checks the GPU, downloads every dataset, and caches them under `data/`.

**Kernel:** select the project's `.venv` kernel ("Python 3.12 (open-data-c-vlm)"), not a system Python.

| Dataset | Used in | Stored at |
| --- | --- | --- |
| CIFAR-10 (60,000 images, 10 classes) | C1 | `data/cifar10/` |
| Flickr30k 1k test split (1,000 images × 5 captions) | C2, C4 | `data/flickr1k/` |
| COCO128 (128 COCO images with box labels) | C3, C4 | Ultralytics datasets folder |
| Two sample photos | C3 | `data/samples/` |
| Three street videos: people, cars, mixed (Intel IoT, CC BY 4.0) | C5 | `data/videos/` |

In [1]:
# Make the project folder importable (works from notebooks/ or the project root)
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

Project root: E:\Users\risaev\Documents\open-data-c-vlm\open-data-c-vlm


In [ ]:
INSTALL = True          # set False after the first run
import sysconfig
IN_VENV = sys.prefix != sys.base_prefix
LOCKED = (pathlib.Path(sysconfig.get_path("stdlib")) / "EXTERNALLY-MANAGED").exists()   # PEP 668
if INSTALL and LOCKED and not IN_VENV:
    raise RuntimeError(
        f"This kernel runs {sys.executable}, a system/uv-managed Python that refuses pip installs.\n"
        "Pick the project kernel instead (top right in VS Code / Jupyter): '.venv' or 'Python 3.12 (open-data-c-vlm)'.\n"
        "Create it once from the project folder:  uv venv .venv --python 3.12  then  "
        "uv pip install --python .venv -r requirements.txt ipykernel")
if INSTALL:
    %pip install -q -r {ROOT}/requirements.txt

## 1. Environment check

In [5]:
import platform, torch, transformers, torchvision
from vlmkit.common import get_device, out_dir, save_json
from vlmkit import data

DEVICE = get_device()
env = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "transformers": transformers.__version__,
    "device": DEVICE,
}
if DEVICE == "cuda":
    p = torch.cuda.get_device_properties(0)
    env["gpu"] = f"{p.name}, {p.total_memory / 1e9:.1f} GB"
try:
    import ultralytics
    env["ultralytics"] = ultralytics.__version__
except ImportError:
    env["ultralytics"] = "missing"
env

ModuleNotFoundError: No module named 'torch'

## 2. CIFAR-10 (for C1)

In [ ]:
from vlmkit.common import show_grid

train, test = data.cifar10(train=True), data.cifar10(train=False)
print(f"train: {len(train)}  test: {len(test)}  classes: {test.classes}")
show_grid([test[i][0].resize((96, 96)) for i in range(10)],
          [test.classes[test[i][1]] for i in range(10)], cols=10, size=1.4)

## 3. Flickr30k 1k test split (for C2 and C4)

Images are written to `data/flickr1k/images/` and captions to `data/flickr1k/captions.csv`, so later notebooks work offline.

In [ ]:
import pandas as pd
from PIL import Image

data.prepare_flickr1k()
items = data.load_flickr1k()
lengths = pd.Series([len(c.split()) for it in items for c in it["captions"]])
print(f"{len(items)} images, {len(lengths)} captions; words per caption: "
      f"median {lengths.median():.0f}, 90th pct {lengths.quantile(.9):.0f}")
show_grid([Image.open(it["path"]) for it in items[:4]],
          [it["captions"][0][:60] for it in items[:4]], cols=4, size=3.2)

## 4. COCO128 (for C3 and C4) and sample photos

In [ ]:
names, coco = data.coco128()
counts = pd.Series({k: sum(it["counts"].get(k, 0) for it in coco) for k in names.values()})
print(f"{len(coco)} images; objects of the 10 most frequent classes:")
print(counts.sort_values(ascending=False).head(10).to_string())
person_hist = pd.Series([it["counts"].get("person", 0) for it in coco]).value_counts().sort_index()
print("\nimages by number of people (used for the C4 counting test):")
print(person_hist.to_string())
samples = data.sample_images()
samples

## 5. Street videos with people and cars (for C5)

Three short clips from the Intel IoT DevKit sample videos (CC BY 4.0), saved to `data/videos/`.

In [ ]:
vids = data.videos()
for name, p in vids.items():
    info = data.video_info(p)
    print(f"{name:7s} {p.name:36s} {info['seconds']:5.1f} s  {info['fps']:.1f} fps  {info['width']}x{info['height']}")
first = {name: Image.open(data.extract_keyframes(p, every_s=5.0)[1]["path"]) for name, p in vids.items()}
show_grid(list(first.values()), [f"video: {n}" for n in first], cols=3, size=3.5)

## 7. Optional: pre-download model weights

SmolVLM2-2.2B is about 4.5 GB (used on GPUs with 8 GB or more); SmolVLM2-500M is about 1 GB (used on smaller GPUs). Pre-downloading avoids waiting during the session.

In [ ]:
PREFETCH = False        # set True to download all model weights now
if PREFETCH:
    from huggingface_hub import snapshot_download
    from ultralytics import YOLO, YOLOWorld
    for repo in ["openai/clip-vit-base-patch32", "HuggingFaceTB/SmolVLM2-2.2B-Instruct",
                 "HuggingFaceTB/SmolVLM2-500M-Video-Instruct"]:
        print("downloading", repo); snapshot_download(repo)
    YOLO("yolo11s.pt"); YOLOWorld("yolov8s-worldv2.pt").set_classes(["person"])
    print("done")

In [ ]:
manifest = {
    "env": env,
    "cifar10": {"train": len(train), "test": len(test)},
    "flickr1k": {"images": len(items), "captions": int(len(lengths))},
    "coco128": {"images": len(coco), "objects": int(counts.sum())},
    "samples": {k: str(v) for k, v in samples.items()},
    "videos": {k: str(v) for k, v in vids.items()},
}
save_json(manifest, out_dir("00_setup") / "data_manifest.json")
print("Saved outputs/00_setup/data_manifest.json")

## 8. Try it live: browse the data

Pick a dataset, slide through it, press **Run**.

In [ ]:
from vlmkit.common import live

def browse(dataset, index):
    """Show 8 consecutive items of a dataset with their labels / captions."""
    if dataset == "CIFAR-10":
        ims = [test[i][0].resize((128, 128)) for i in range(index, index + 8)]
        titles = [test.classes[test[i][1]] for i in range(index, index + 8)]
    elif dataset == "Flickr 1k":
        sel = items[index % len(items):][:8]
        ims, titles = [Image.open(it["path"]) for it in sel], [it["captions"][0][:50] for it in sel]
    elif dataset == "COCO128":
        sel = coco[index % len(coco):][:8]
        ims = [Image.open(it["path"]) for it in sel]
        titles = [", ".join(f"{n} {k}" for k, n in sorted(it["counts"].items(), key=lambda x: -x[1])[:3]) for it in sel]
    else:                                                     # videos: 8 keyframes across the clip
        ks = data.extract_keyframes(vids[dataset.split()[-1]], every_s=5.0)
        ims, titles = [Image.open(k["path"]) for k in ks[:8]], [f"t = {k['t']:.0f} s" for k in ks[:8]]
    show_grid(ims, titles, cols=4, size=3)

live(browse, dataset=["CIFAR-10", "Flickr 1k", "COCO128"] + [f"video {v}" for v in vids], index=(0, 120, 8, 0))